# 12. 툴 콜링 — 오픈 모델 vs API

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/acho98/gunyang-ai-api/blob/main/labs/13_tool_calling_open_vs_api.ipynb)

> 🧭 **스토리** — 규정 도우미가 "과장 부산 2박 3일이면 얼마?"에 답하려면 규정을 **찾고**, 숫자를 **계산**해야 한다. 모델에게 도구라는 손을 달아 준다. 작은 오픈 모델도 같은 손을 쓸 수 있을까?
>
> 📖 **이론 연결** — 이론2 p69~74 (툴 콜링 5단계·오픈 모델 툴 콜링·크기별 성능·대형 모델 실행 조건) · ⏱ **15분** · 🖥 **GPU 권장**
>
> GPU(T4)가 있으면 빠릅니다. 없어도 CPU로 돌아갑니다(느림).
> 💡 **앞 노트북을 열어 둔 상태라면** 그 노트북에서 `런타임 → 런타임 연결 해제 및 삭제`를 먼저 하세요. 무료 코랩은 GPU 세션을 하나만 허용합니다.

**셀 배지**: ⚙️ 준비 · 🔵 따라 실행 · 🟡 바꿔서 실행 · 🤔 생각해보기 · 🟠 심화(시간 남을 때)

## 0. 준비 ⚙️
설치는 세 가지입니다. OpenAI SDK, 검색용 `rank_bm25`, 한국어 형태소 분석기 `kiwipiepy`(Java 불필요, 휠 11MB). 임베딩 모델은 받지 않습니다. 검색은 BM25 키워드 검색으로 가볍게 합니다.

In [ ]:
%pip -q install openai rank_bm25 kiwipiepy

In [ ]:
# ⚙️ 실습 데이터 로더 — 깃허브(gunyang-data/genai)에서 자동으로 받아옵니다. 업로드 불필요.
import os, json, urllib.request
from pathlib import Path
DATA_BASE = os.environ.get("GENAI_BASE", "https://raw.githubusercontent.com/acho98/gunyang-data/main/genai/")
CACHE = Path("genai_data"); CACHE.mkdir(exist_ok=True)

def fetch(name: str) -> str:
    """파일 이름(예: 'hanbit_rules.pdf', 'scan/p2.png') → 로컬 경로. URL이면 내려받아 캐시."""
    if DATA_BASE.startswith("http"):
        dst = CACHE / name
        if not dst.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(DATA_BASE + name, dst)
        return str(dst)
    return str(Path(DATA_BASE) / name)

def load_json(name: str):
    with open(fetch(name), encoding="utf-8") as f:
        return json.load(f)

def load_text(name: str) -> str:
    with open(fetch(name), encoding="utf-8") as f:
        return f.read()

# 한글 폰트(차트용) — gunyang-data의 BMDOHYEON.ttf 를 자동으로 받아 적용. 실패해도 진행.
def set_korean_font():
    import matplotlib.pyplot as plt
    from matplotlib import font_manager as fm
    font_url = "https://raw.githubusercontent.com/acho98/gunyang-data/main/BMDOHYEON.ttf"
    dst = CACHE / "BMDOHYEON.ttf"
    try:
        if not dst.exists():
            local = Path(DATA_BASE).parent / "BMDOHYEON.ttf" if not DATA_BASE.startswith("http") else None
            if local and local.exists():
                dst.write_bytes(local.read_bytes())
            else:
                urllib.request.urlretrieve(font_url, dst)
        fm.fontManager.addfont(str(dst))
        plt.rcParams["font.family"] = fm.FontProperties(fname=str(dst)).get_name()
        plt.rcParams["axes.unicode_minus"] = False
    except Exception as e:
        print("(안내) 한글 폰트 설정 실패 — 영문 라벨은 정상:", e)
set_korean_font()
print("데이터 소스:", DATA_BASE)

In [ ]:
# ⚙️ OpenAI API 키 입력 — 수업에서 나눠 준 키를 붙여 넣으세요. (화면에 표시되지 않습니다)
# 규칙: 키를 코드에 직접 적지 말 것 · 출력하지 말 것 · 수업 후 키는 폐기됩니다.
import os
from getpass import getpass
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API 키: ").strip()
MODEL_API = "gpt-6-luna"        # 수업용 소형 모델 ($0.10 / $0.50 per 1M tokens). 바뀌면 여기만 수정.
print("키 입력 완료 · 모델:", MODEL_API)

In [ ]:
# ⚙️ API 오류를 한국어로 설명하는 헬퍼 — 키가 없거나 한도에 걸려도 노트북이 멈추지 않게 합니다
import openai
def explain_error(e):
    if isinstance(e, openai.AuthenticationError):
        print("🔑 401 인증 실패 — 키가 없거나 틀렸습니다. 키 셀을 다시 실행하세요. (이 셀의 OpenAI 부분은 건너뜁니다)")
    elif isinstance(e, openai.RateLimitError):
        if "project_spend_limit_exceeded" in str(e):
            print("💸 429 project_spend_limit_exceeded — 수업용 프로젝트의 지출 한도에 도달했습니다. 강사에게 알려 주세요.")
        else:
            print("⏳ 429 속도 제한 — 잠시 뒤 다시 실행하세요.")
    elif isinstance(e, openai.APIConnectionError):
        print("🌐 연결 실패 — 서버 주소나 네트워크를 확인하세요:", str(e)[:120])
    else:
        print("⚠️", type(e).__name__, str(e)[:200])

## 1. 도구 3개 만들기 🔵
도구는 **그냥 파이썬 함수**입니다. 모델은 함수를 직접 실행하지 못합니다. "이 함수를 이 인자로 불러 달라"고 **요청**만 합니다. 실행은 우리 코드가 합니다(이론2 p69).

| 도구 | 하는 일 |
|---|---|
| `calc(expression)` | 사칙연산. `eval` 대신 `ast`로 숫자·연산자만 허용(안전) |
| `today()` | 오늘 날짜와 요일 |
| `search_rules(query, k=3)` | 규정집 13개 청크에서 BM25 키워드 검색. 한국어는 형태소로 쪼개서 셉니다 |

In [ ]:
import ast, operator, json, datetime
from kiwipiepy import Kiwi
from rank_bm25 import BM25Okapi

# --- calc: 허용한 연산자만 계산합니다. eval(expression)은 임의 코드 실행 위험이 있어 쓰지 않습니다.
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
       ast.FloorDiv: operator.floordiv, ast.Mod: operator.mod, ast.Pow: operator.pow, ast.USub: operator.neg}
def _eval(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in OPS:
        return OPS[type(node.op)](_eval(node.left), _eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in OPS:
        return OPS[type(node.op)](_eval(node.operand))
    raise ValueError("허용하지 않는 수식")
def calc(expression: str) -> dict:
    expr = expression.replace(",", "").replace("원", "").replace("×", "*").replace("x", "*")
    try:
        return {"expression": expression, "result": _eval(ast.parse(expr, mode="eval").body)}
    except Exception as e:
        return {"expression": expression, "error": str(e)}

# --- today
def today() -> dict:
    d = datetime.date.today()
    return {"today": d.isoformat(), "weekday": "월화수목금토일"[d.weekday()] + "요일"}

# --- search_rules: 정답 청크 13개 위에서 BM25 검색
CHUNKS = load_json("hanbit_chunks_gt.json")
kiwi = Kiwi()
def ko_tokens(text):  # 형태소 중 명사·동사·숫자 등 내용어만 남깁니다
    return [t.form for t in kiwi.tokenize(text) if t.tag[0] in "NVSXM" and t.tag not in ("SF", "SP", "SS", "SE", "SO", "SW")]
bm25 = BM25Okapi([ko_tokens(c["text"]) for c in CHUNKS])
def search_rules(query: str, k: int = 3) -> list:
    scores = bm25.get_scores(ko_tokens(query))
    top = sorted(range(len(CHUNKS)), key=lambda i: -scores[i])[:k]
    return [{"id": CHUNKS[i]["id"], "section": CHUNKS[i]["section_title"], "text": CHUNKS[i]["text"]} for i in top]

FUNCS = {"calc": calc, "today": today, "search_rules": search_rules}   # 이름 → 실제 함수
print(calc("30,000*3 + 110,000*2"))
print(calc("__import__('os').system('ls')"))   # 막혀야 정상
print(today())
for h in search_rules("과장 국내 출장 일비 숙박비"):
    print(h["id"], "|", h["text"][:60].replace("\n", " "))

💡 **해석** — `calc`는 위험한 수식을 `허용하지 않는 수식`으로 막습니다. 모델이 만든 문자열을 그대로 `eval`하면 안 되는 이유입니다(이론2 p69 실행 안전성). `search_rules`는 `s2_t1_domestic`(표 1. 국내 출장비)을 1위로 찾습니다. 질문에 답할 재료가 이 표에 있습니다.

## 2. 도구 설명서(스키마) — Responses API 형식 🔵
모델은 함수 코드를 보지 못합니다. **이름·설명·파라미터 스키마**만 봅니다. 그래서 설명이 곧 사용 설명서입니다.
- `strict: True` → 모델이 만드는 인자가 스키마를 반드시 지킵니다(Structured Outputs와 같은 원리).
- strict 모드에서는 모든 속성을 `required`에 넣고 `additionalProperties: False`를 붙여야 합니다(이론2 p75).
- Responses API는 **평평한(flat)** 형식입니다. 뒤에서 볼 Chat Completions는 `function` 안에 한 번 더 감쌉니다.

In [ ]:
TOOLS = [
    {"type": "function", "name": "search_rules",
     "description": "한빛테크 임직원 출장·복리후생 규정집에서 질문과 관련된 조항과 표를 검색한다. 금액·일수·조건 등 규정 사실이 필요하면 반드시 먼저 호출한다.",
     "parameters": {"type": "object",
                    "properties": {"query": {"type": "string", "description": "검색어. 예: '과장 국내 출장 일비 숙박비'"}},
                    "required": ["query"], "additionalProperties": False},
     "strict": True},
    {"type": "function", "name": "calc",
     "description": "사칙연산 수식을 계산한다. 금액 합계·곱셈은 직접 암산하지 말고 이 도구를 쓴다.",
     "parameters": {"type": "object",
                    "properties": {"expression": {"type": "string", "description": "파이썬 수식. 예: '30000*3 + 110000*2'"}},
                    "required": ["expression"], "additionalProperties": False},
     "strict": True},
    {"type": "function", "name": "today",
     "description": "오늘 날짜(YYYY-MM-DD)와 요일을 돌려준다.",
     "parameters": {"type": "object", "properties": {}, "required": [], "additionalProperties": False},
     "strict": True},
]
SYSTEM = ("너는 한빛테크 규정 도우미다. 규정 사실은 search_rules로 찾은 근거로만 답하고, 계산은 calc로 한다. "
          "답의 끝에 근거 청크 id를 [s2_t1_domestic]처럼 적는다. 근거가 없으면 모른다고 답한다.")
QUESTION = "과장이 부산에 2박 3일 출장 가면 일비와 숙박비 상한 합계는?"
print(json.dumps(TOOLS[0], ensure_ascii=False, indent=1)[:400])

## 3. 툴콜링 4단계를 한 단계씩 🔵
① 스키마와 질문 전송 → ② 모델이 `function_call` 반환 → ③ 우리가 실행 → ④ `function_call_output`으로 결과 전달 → 최종 답.
먼저 ①②만 해서 모델이 **무엇을 요청하는지** 봅니다.

In [ ]:
from openai import OpenAI
client = OpenAI()
input_list = [{"role": "user", "content": QUESTION}]
r1 = None
try:
    r1 = client.responses.create(model=MODEL_API, instructions=SYSTEM, input=input_list, tools=TOOLS,
                                 reasoning={"effort": "low"})
    for item in r1.output:
        print(item.type, "→", getattr(item, "name", ""), getattr(item, "arguments", ""), getattr(item, "call_id", ""))
except Exception as e:
    explain_error(e)

💡 **해석** — 출력에 `function_call` 항목이 보이면 모델이 "규정집을 검색해 달라"고 **요청**한 것입니다. `name`은 도구 이름, `arguments`는 **JSON 문자열**, `call_id`는 이 요청의 번호표입니다. 결과를 돌려줄 때 같은 `call_id`를 붙여야 모델이 짝을 맞춥니다. `reasoning` 항목이 함께 보일 수 있습니다. 모델이 도구를 고르기 전에 생각한 흔적입니다.

③ 실행하고 ④ 결과를 돌려줍니다. `input_list += r1.output`으로 모델의 요청(과 reasoning 항목)을 대화에 그대로 붙인 뒤 결과를 덧붙입니다.

In [ ]:
if r1 is not None:
    input_list += r1.output                       # 모델이 한 요청을 대화 기록에 그대로 남깁니다
    for item in r1.output:
        if item.type == "function_call":
            args = json.loads(item.arguments)     # 문자열 → dict
            result = FUNCS[item.name](**args)     # ③ 우리가 실행
            input_list.append({"type": "function_call_output", "call_id": item.call_id,
                               "output": json.dumps(result, ensure_ascii=False)})   # ④ 결과 전달
            print(f"실행: {item.name}({args}) → {str(result)[:120]}…")
    try:
        r2 = client.responses.create(model=MODEL_API, instructions=SYSTEM, input=input_list, tools=TOOLS,
                                     reasoning={"effort": "low"})
        print([(it.type, getattr(it, "name", "")) for it in r2.output])
        print("답:", r2.output_text)
    except Exception as e:
        explain_error(e)
else:
    print("앞 셀의 API 호출이 실패해서 건너뜁니다.")

💡 **해석** — 두 번째 응답에 또 `function_call`(대개 `calc`)이 있으면 모델이 **검색 결과를 읽고 계산을 요청**한 것입니다. 한 번에 끝나지 않습니다. 그래서 다음 절처럼 **반복문**이 필요합니다. 기대 정답은 일비 30,000원×3일 + 숙박비 110,000원×2박 = **310,000원**입니다.

## 4. 반복문으로 감싸기 — 에이전트 루프의 뼈대 🔵
`function_call`이 없을 때까지 ②~④를 되풀이합니다. 무한 반복을 막으려고 `max_turns=5`를 둡니다(이론2 p69). 이 10줄 남짓이 에이전트의 핵심입니다.

In [ ]:
def run_tools(question, tools=TOOLS, funcs=FUNCS, max_turns=5, verbose=True):
    """Responses API 툴 루프. 반환: {'calls': [(이름, 인자문자열)], 'final': 최종 답}"""
    inp, calls = [{"role": "user", "content": question}], []
    for turn in range(1, max_turns + 1):
        r = client.responses.create(model=MODEL_API, instructions=SYSTEM, input=inp, tools=tools,
                                    reasoning={"effort": "low"})
        fcs = [it for it in r.output if it.type == "function_call"]
        if not fcs:
            if verbose: print(f"[{turn}] 최종 답 → {r.output_text}")
            return {"calls": calls, "final": r.output_text}
        inp += r.output
        for it in fcs:
            calls.append((it.name, it.arguments))
            try:
                result = funcs[it.name](**json.loads(it.arguments))
            except Exception as e:                    # 없는 도구·잘못된 인자도 모델에게 알려 줍니다
                result = {"error": f"{type(e).__name__}: {e}"}
            inp.append({"type": "function_call_output", "call_id": it.call_id,
                        "output": json.dumps(result, ensure_ascii=False)})
            if verbose: print(f"[{turn}] {it.name}({it.arguments}) → {str(result)[:90]}…")
    return {"calls": calls, "final": "(max_turns 초과)"}

try:
    out = run_tools(QUESTION)
except Exception as e:
    explain_error(e)

💡 **해석** — 로그에서 순서를 확인하세요. 보통 `[1] search_rules` → `[2] calc` → `[3] 최종 답`입니다. 규정 검색과 계산이 **두 단계로 이어진 호출**입니다. 모델이 표의 숫자를 읽어 수식을 직접 만들었다는 점이 중요합니다. 프로그래머는 "과장이면 30,000" 같은 규칙을 한 줄도 쓰지 않았습니다.

## 5. 같은 손을 오픈 모델에게 — Ollama 🔵
이제 같은 도구를 로컬 오픈 모델에게 줍니다. Ollama는 OpenAI 호환 `/v1/chat/completions`에서 `tools`를 지원합니다.
- `qwen3:4b` (2.5GB, Ollama 공식 "tools" 배지) — 주력 오픈 모델
- `EXAONE-4.0-1.2B` Q4_K_M GGUF (812MB) — 노트북 05에서 쓴 한국어 소형 모델

코랩 T4 기준 설치·다운로드에 1~2분 걸립니다.

In [ ]:
# ⚙️ Ollama 설치 + 서버 실행 (코랩 리눅스 VM 안에서). 첫 실행 1분 안팎.
import subprocess, shutil, time, os, urllib.request
if not shutil.which("ollama"):
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
# 서버를 백그라운드로 띄우고 준비될 때까지 기다립니다
if not os.environ.get("_OLLAMA_STARTED"):
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    os.environ["_OLLAMA_STARTED"] = "1"
for _ in range(30):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2); break
    except Exception:
        time.sleep(1)
print("Ollama 준비 완료:", subprocess.run(["ollama", "--version"], capture_output=True, text=True).stdout.strip())

In [ ]:
# ⚙️ 모델 받기 — GGUF(Q4_K_M 양자화) 를 허깅페이스에서 Ollama로 바로 당겨옵니다
MODEL_LOCAL = "qwen3:4b"
import subprocess, time
t0 = time.time()
r = subprocess.run(["ollama", "pull", MODEL_LOCAL], capture_output=True, text=True)
print(r.stdout[-300:] if r.returncode == 0 else r.stderr[-500:])
print(f"pull 소요 {time.time()-t0:.0f}초")
print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)

In [ ]:
# ⚙️ 모델 받기 — GGUF(Q4_K_M 양자화) 를 허깅페이스에서 Ollama로 바로 당겨옵니다
MODEL_LOCAL = "hf.co/LGAI-EXAONE/EXAONE-4.0-1.2B-GGUF:Q4_K_M"
import subprocess, time
t0 = time.time()
r = subprocess.run(["ollama", "pull", MODEL_LOCAL], capture_output=True, text=True)
print(r.stdout[-300:] if r.returncode == 0 else r.stderr[-500:])
print(f"pull 소요 {time.time()-t0:.0f}초")
print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)

In [ ]:
LOCAL_MODELS = {"qwen3:4b": "qwen3:4b", "EXAONE-1.2B": "hf.co/LGAI-EXAONE/EXAONE-4.0-1.2B-GGUF:Q4_K_M"}
local = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")   # 05에서 본 '같은 클라이언트, 다른 주소'

# Chat Completions 형식은 function 안에 한 번 더 감쌉니다(nested). strict는 Ollama가 쓰지 않으므로 뺍니다.
CHAT_TOOLS = [{"type": "function",
               "function": {"name": t["name"], "description": t["description"], "parameters": t["parameters"]}}
              for t in TOOLS]
print(json.dumps(CHAT_TOOLS[1], ensure_ascii=False)[:300])

💡 **해석** — 형식만 다르고 내용은 같습니다. Responses는 `{"type":"function","name":…}`, Chat Completions는 `{"type":"function","function":{"name":…}}`입니다. 결과를 돌려줄 때도 다릅니다. Responses는 `function_call_output` 항목, Chat Completions는 `{"role":"tool","tool_call_id":…}` 메시지를 씁니다(이론2 p71).

In [ ]:
LOCAL_OPTS = {"reasoning_effort": "none"}   # 생각 모드 끄기 — EXAONE에는 이것이 듣습니다(think=False는 무시됨)
def strip_think(text):
    """생각 글이 본문에 섞여 '…</think>' 로 끝나면 그 뒤의 진짜 답만 남깁니다(대비책)."""
    return (text or "").split("</think>")[-1].strip()

def run_tools_chat(cli, model, question, tools=CHAT_TOOLS, funcs=FUNCS, max_turns=5, verbose=True, opts=None):
    """Chat Completions 툴 루프(Ollama). 반환: {'calls': [(이름, 인자문자열)], 'final': 최종 답}"""
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    calls = []
    for turn in range(1, max_turns + 1):
        resp = cli.chat.completions.create(model=model, messages=msgs, tools=tools, **(opts or LOCAL_OPTS))
        msg = resp.choices[0].message
        content = strip_think(msg.content)
        if not msg.tool_calls:
            if verbose: print(f"[{turn}] 최종 답 → {content[:200]}")
            return {"calls": calls, "final": content}
        msgs.append({"role": "assistant", "content": content,
                     "tool_calls": [{"id": tc.id, "type": "function",
                                     "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                                    for tc in msg.tool_calls]})
        for tc in msg.tool_calls:
            calls.append((tc.function.name, tc.function.arguments))
            try:
                args = json.loads(tc.function.arguments or "{}")
            except json.JSONDecodeError:
                args = None
            try:
                result = funcs[tc.function.name](**args) if args is not None else {"error": "arguments가 JSON이 아님"}
            except Exception as e:
                result = {"error": f"{type(e).__name__}: {e}"}
            msgs.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(result, ensure_ascii=False)})
            if verbose: print(f"[{turn}] {tc.function.name}({tc.function.arguments}) → {str(result)[:90]}…")
    return {"calls": calls, "final": "(max_turns 초과)"}

for name, model in LOCAL_MODELS.items():
    print(f"===== {name} =====")
    try:
        run_tools_chat(local, model, QUESTION)
    except Exception as e:
        explain_error(e)

💡 **해석** — 강사 환경(Ollama 0.35, Apple M5 Max) 시험에서 `qwen3:4b`는 `search_rules`로 표 1을 찾고 310,000원을 답했습니다. `calc`를 부르지 않고 암산으로 끝내는 경우도 있습니다. 지시문에 "계산은 calc로"라고 썼는데도 그렇습니다. 작은 모델은 지시를 덜 지킵니다. EXAONE 1.2B는 Ollama 모델 정보에 "tools" 기능이 표시되는데도 도구를 부르지 않고 바로 줄글로 답했습니다. 한 번은 우연일 수 있습니다. 그래서 다음 절에서 **10번씩** 돌려 셉니다.

## 6. 10회 반복 성공률 표 🔵
같은 질문을 백엔드마다 10번 돌려 세 가지를 셉니다.
- **도구 선택**: `search_rules`를 한 번이라도 불렀는가(규정 질문이니 검색이 맞는 도구)
- **인자 JSON 유효**: 도구를 불렀고, 모든 `arguments`가 JSON으로 읽히는가
- **정답**: 최종 답에 310,000(또는 31만)이 들어 있는가

키가 있으면 `gpt-6-luna`도 4절의 Responses 루프로 함께 잽니다. 시간이 부족하면 `N_RUNS = 3`으로 줄이세요.

In [ ]:
import time, pandas as pd
N_RUNS = 10
def valid_json(s):
    try:
        return isinstance(json.loads(s or "{}"), dict)
    except json.JSONDecodeError:
        return False
def is_correct(text):
    t = text.replace(" ", "")
    return ("310,000" in t) or ("310000" in t) or ("31만" in t)

# 이름: (모델, 생각 끄기 옵션). qwen3는 think=False 방식도 함께 비교합니다.
backends = {"qwen3:4b": (LOCAL_MODELS["qwen3:4b"], {"reasoning_effort": "none"}),
            "qwen3:4b (think=False)": (LOCAL_MODELS["qwen3:4b"], {"extra_body": {"think": False}}),
            "EXAONE-1.2B": (LOCAL_MODELS["EXAONE-1.2B"], {"reasoning_effort": "none"})}
api_ok = True
try:
    client.responses.create(model=MODEL_API, input="ping", max_output_tokens=16, reasoning={"effort": "none"})
except Exception as e:
    api_ok = False; explain_error(e); print("→ 표에서 gpt-6-luna 행은 빠집니다.")

rows = []
for name in list(backends) + (["gpt-6-luna"] if api_ok else []):
    cli, (model, opts) = (client, (MODEL_API, None)) if name == "gpt-6-luna" else (local, backends[name])
    tool_ok = json_ok = ans_ok = 0; t0 = time.time()
    for i in range(N_RUNS):
        try:   # API는 앞에서 만든 Responses 루프, 오픈 모델은 Chat Completions 루프
            o = run_tools(QUESTION, verbose=False) if name == "gpt-6-luna" else run_tools_chat(cli, model, QUESTION, verbose=False, opts=opts)
        except Exception as e:
            o = {"calls": [], "final": f"ERR {type(e).__name__}"}
        tool_ok += any(c[0] == "search_rules" for c in o["calls"])
        json_ok += bool(o["calls"]) and all(valid_json(c[1]) for c in o["calls"])
        ans_ok += is_correct(o["final"])
    rows.append({"백엔드": name, "도구 선택": f"{tool_ok}/{N_RUNS}", "인자 JSON 유효": f"{json_ok}/{N_RUNS}",
                 "정답 310,000": f"{ans_ok}/{N_RUNS}", "1회 평균(초)": round((time.time() - t0) / N_RUNS, 1)})
    print(rows[-1])
display(pd.DataFrame(rows))

💡 **해석** — 강사 환경(Apple M5 Max, Ollama 0.35, 2026-10-12) 실측입니다. 코랩 T4에서는 시간이 다르고, 횟수도 실행마다 조금씩 달라집니다.

| 백엔드 | 도구 선택 | 인자 JSON 유효 | 정답 310,000 |
|---|---|---|---|
| qwen3:4b (`reasoning_effort="none"`) | 10/10 | 10/10 | 10/10 |
| qwen3:4b (`think=False`) | 10/10 | 10/10 | 10/10 |
| EXAONE-1.2B Q4 (`reasoning_effort="none"`) | 0/10 | 0/10 | 0/10 |
| gpt-6-luna | 키가 있을 때 측정 | | |

- **qwen3:4b**는 10번 모두 검색 도구를 골랐고, 인자도 모두 유효한 JSON이었고, 10번 모두 310,000원을 맞혔습니다. 이 질문은 2~3단계라서 4B 모델도 감당합니다. 1회 평균은 강사 환경에서 30~35초였습니다.
- **생각 끄기 옵션은 모델마다 다르게 듣습니다.** EXAONE은 `reasoning_effort="none"`이면 깔끔하게 답만 내고, `think=False`는 무시하고 생각 글을 본문에 섞습니다. qwen3:4b는 반대로 `think=False`가 깔끔하고 조금 빠릅니다. `reasoning_effort="none"`을 주면 생각 글이 본문에 섞여 `</think>`로 끝납니다. 그래서 루프는 `reasoning_effort="none"`을 기본으로 보내고, 본문에서 마지막 `</think>`까지를 잘라 내는 `strip_think`를 대비책으로 둡니다. 정답률은 두 방식이 같았습니다.
- **EXAONE 1.2B Q4**는 Ollama 경로에서 도구를 **한 번도 부르지 않았습니다(0/10)**. 생각 옵션을 바꿔도 같았습니다. 대신 규정을 모르는 채로 "일비 상한 30만 원" 같은 숫자를 지어냈습니다. Ollama 모델 정보에 "tools"가 표시되어도 실제로 호출을 내는지는 별개입니다. 이 경로에서는 "툴콜링 지원 안 함"으로 보고, EXAONE의 툴 템플릿은 9절 🟠의 transformers 경로로 확인합니다.

공개 벤치마크도 같은 이야기를 합니다(이론2 p72).

| 벤치마크 | 작은 모델 | 큰 모델 |
|---|---|---|
| BFCL-v3(단발 함수 호출) | EXAONE 4.0 1.2B **55.7** | EXAONE 4.0 32B **65.2** |
| BFCL-v3 | Qwen3-4B-Instruct-2507 **61.9** | Qwen3-30B-A3B **65.1** |
| Tau-bench Retail(다단계 에이전트) | EXAONE 4.0 1.2B **21.7** | EXAONE 4.0 32B **55.9** |

**단발 호출 점수는 비슷한데, 여러 단계를 이어 가는 일에서 차이가 크게 벌어집니다.** 오늘 문제는 검색 → 계산 → 답의 3단계라서 이 차이가 그대로 보입니다. gpt-oss-20b(약 14GB)나 qwen3:30b(약 19GB)는 툴콜링이 강하지만 무료 T4(15GB)에 올라가지 않습니다.

## 7. 도구 설명이 나쁘면? — 모호한 스키마 실험 🔵
같은 함수에 이름을 `tool_a`/`tool_b`/`tool_c`로, 설명을 "도구입니다."로 바꿔 봅니다. 함수 코드는 그대로입니다. 모델이 보는 **설명서만** 나빠졌습니다.

In [ ]:
VAGUE = [{"type": "function", "function": {"name": n, "description": "도구입니다.",
           "parameters": {"type": "object", "properties": {"x": {"type": "string"}}, "required": ["x"]}}}
         for n in ["tool_a", "tool_b", "tool_c"]]
VAGUE_FUNCS = {"tool_a": lambda x: search_rules(x), "tool_b": lambda x: calc(x), "tool_c": lambda x: today()}
for i in range(3):
    print(f"--- qwen3:4b, 모호한 도구 #{i+1} ---")
    try:
        o = run_tools_chat(local, LOCAL_MODELS["qwen3:4b"], QUESTION, tools=VAGUE, funcs=VAGUE_FUNCS, verbose=True)
        print("호출 순서:", [c[0] for c in o["calls"]], "| 정답 포함:", is_correct(o["final"]))
    except Exception as e:
        explain_error(e)

💡 **해석** — 강사 환경에서 이 셀을 세 번 돌렸고(각 3회), 같은 qwen3:4b가 **9번 모두 실패**했습니다. 날짜 도구 `tool_c`를 검색 도구로 착각해 부르기(6회), 도구 없이 "모른다"로 끝내기(2회), 도구 없이 700,000이라는 틀린 숫자를 지어내고 근거 id까지 붙이기(1회)였습니다. 좋은 설명에서는 10/10이었던 모델입니다. 특히 **근거 id를 붙인 틀린 답**은 그럴듯해 보여서 더 위험합니다.

도구 설명은 **모델을 위한 프롬프트**입니다. 무엇을 하는지, 언제 쓰는지, 인자 예시를 적으세요. Anthropic의 「Writing tools for agents」도 같은 원칙을 말합니다(이론2 p77).

## 8. 바꿔서 실행 🟡
해외 출장 질문에 답하려면 달러를 원화로 바꿔야 합니다. `convert_currency(amount, to)` 도구를 추가해 보세요. 환율은 수업용 고정값입니다.
1. 아래 셀을 그대로 실행해 결과를 봅니다.
2. `QUESTION2`를 바꾸거나 `RATES`에 통화를 추가해 다시 실행합니다.

In [ ]:
RATES = {"KRW": 1400.0, "JPY": 150.0, "EUR": 0.92, "USD": 1.0}   # 1 USD당 (수업용 고정 환율)
def convert_currency(amount: float, to: str) -> dict:
    to = to.upper()
    if to not in RATES:
        return {"error": f"지원 통화: {list(RATES)}"}
    return {"usd": amount, "to": to, "result": round(amount * RATES[to], 2)}

TOOLS2 = TOOLS + [{"type": "function", "name": "convert_currency",
    "description": "미국 달러(USD) 금액을 다른 통화로 바꾼다. 해외 출장비(USD 기준)를 원화로 안내할 때 쓴다.",
    "parameters": {"type": "object",
                   "properties": {"amount": {"type": "number", "description": "USD 금액"},
                                  "to": {"type": "string", "enum": list(RATES), "description": "바꿀 통화 코드"}},
                   "required": ["amount", "to"], "additionalProperties": False},
    "strict": True}]
FUNCS2 = {**FUNCS, "convert_currency": convert_currency}
CHAT_TOOLS2 = [{"type": "function", "function": {"name": t["name"], "description": t["description"], "parameters": t["parameters"]}}
               for t in TOOLS2]

QUESTION2 = "차장이 유럽으로 3일 출장을 가면 일비 합계는 원화로 얼마인가?"   # ← 바꿔 보세요 (기대: 80×3=240 USD → 336,000원)
try:
    run_tools(QUESTION2, tools=TOOLS2, funcs=FUNCS2)
except Exception as e:
    explain_error(e)
    print("→ 키 없이 로컬 qwen3:4b로 시도합니다.")
    try:
        run_tools_chat(local, LOCAL_MODELS["qwen3:4b"], QUESTION2, tools=CHAT_TOOLS2, funcs=FUNCS2)
    except Exception as e2:
        explain_error(e2)

💡 **해석** — 기대 흐름은 `search_rules`(표 2. 해외 출장비) → `calc("80*3")` → `convert_currency(240, "KRW")` → **336,000원**입니다. 강사 환경에서 키 없이 qwen3:4b로 세 번 돌렸습니다. 한 번은 `convert_currency`까지 불러 336,000원을 맞혔고, 두 번은 240 USD까지 구하고 멈췄습니다. 질문에 "원화로"가 있는데도 마지막 단계를 빼먹은 것입니다. 단계가 늘수록 작은 모델이 흔들리는 모습입니다. `to`에 `enum`을 걸어 두면 모델이 "원화" 같은 엉뚱한 값을 넣지 못합니다. 스키마가 곧 안전장치입니다.

## 9. 🟠 심화 — 오픈 모델 툴콜링의 속살
Ollama가 감춰 주는 일을 직접 해 보고 싶다면, transformers에서 `tokenizer.apply_chat_template(messages, tools=[calc, ...])`로 프롬프트를 만들어 보세요. 모델이 `<tool_call>{...}</tool_call>` **텍스트**를 내고, 그것을 우리가 파싱해 `role: "tool"` 메시지로 돌려줍니다. 이 경로에서는 `arguments`가 dict입니다. OpenAI API는 문자열입니다(이론2 p71). 참고: Hugging Face 문서 「Tool use」(chat_extras).

### 🤔 생각해보기

1) BFCL 점수는 비슷한데 왜 다단계 작업에서 차이가 벌어질까요?

2) 툴콜링을 잘하는 20B+ 오픈 모델을 오늘 코랩에서 못 돌리는 이유는?

<details><summary><b>🤔 생각해보기 — 해설 보기(클릭)</b></summary>

1) 단계마다 성공 확률이 곱해집니다. 한 번 호출 성공률이 90%라도 3단계면 0.9³ ≈ 73%, 5단계면 59%입니다. 작은 모델은 앞 단계 결과(검색된 표)를 읽고 다음 인자를 정확히 만드는 능력, 대화가 길어져도 지시를 지키는 능력이 약합니다. BFCL은 대부분 한 번의 호출을 채점하고, Tau-bench는 여러 턴을 끝까지 채점합니다.

2) 메모리입니다. gpt-oss-20b는 MXFP4로도 약 14GB, qwen3:30b는 Q4로 약 19GB입니다. 무료 T4는 15GB이고 KV 캐시 공간도 필요합니다. 로컬은 4B급까지, 그 위는 API나 더 큰 GPU가 현실적입니다.

</details>

---
### ✅ 이 노트북에서 가져갈 것
- 모델은 도구를 **요청**만 한다. 실행은 우리 코드가 하고, 결과를 `call_id`로 짝지어 돌려준다.
- 에이전트의 뼈대는 `function_call`이 없을 때까지 도는 반복문 + `max_turns`.
- Responses는 flat 스키마·`function_call_output`, Chat Completions는 nested 스키마·`role: tool`.
- 작은 오픈 모델도 툴콜링을 하지만 다단계에서 흔들린다. 반복 측정으로 판단한다.
- 도구 설명이 나쁘면 오호출이 늘어난다. 설명은 모델용 프롬프트다.

> 다음 노트북: **14_agents_sdk — 반복문을 SDK에 맡기고 가드레일·핸드오프 붙이기**